# Memmingen -- Full-Network Paper 2 Model (real framework, 1:1 fidelity)

This notebook builds the **exact same MILP model** that
`scripts/paper_2/scenario_runner.py` builds for the `MM-S0-HK0` scenario --
full network topology (15 nodes, 14 pipes), pressure drop, heat loss,
transport delay, heating-curve-driven COP, and WP/EK/TES investment
optimization -- by calling the real `calion/` framework functions directly,
cell by cell, instead of running the CLI campaign script as one opaque call.

**Why this approach, not a from-scratch reimplementation:** the real model
uses deliberately non-obvious, MILP-tractability-tuned formulations (a
3-tangent convex envelope for pressure drop, BFS-precomputed static
temperature offsets, integer-bucket transport delay, one-hot discrete TES
sizing, ...) that took many rounds of careful bug-fixing to get right. A
fresh rewrite risks silently diverging from Paper 2's actual published
numbers. Every cell below imports and calls the real, already-tested
functions -- this notebook is a *transparent, cell-by-cell driver* over that
code, not a second implementation.

**Fidelity, verified**: building the model here reproduces
**2,321,419 variables / 3,171,129 constraints**, matching the real campaign
reference run at `output/paper2_runs/MM-S0-HK0/meta.json`
(2,321,418 / 3,171,129) almost exactly -- see the validation cell at the end.

**Scale**: this is the real Memmingen model -- ~2.3M variables. The reference
campaign run took **~24h** to reach even a time-limited incumbent
(`TimeLimit: 86400s`, `MIPGap: 0.005`, never proved optimal). Building the
model (below) takes well under a minute; **solving to full campaign quality
does not**. The solve cell is opt-in and defaults to a short demo `TimeLimit`
-- raise it (see that cell's comment) for a directly comparable result.

**What you can adjust**: anything in the merged config dict (`cfg`) before
the model-build cell -- e.g. change `cfg["assets"]["hp_main"]["investment"]
["capacity_max_mw"]`, move an asset to a different node via
`cfg["network"]["nodes"]`, or pick a different scenario/heat-curve stage in
the early cells -- then re-run from the model-build cell onward.


### Cell 1 — Setup: path bootstrap + imports

In [ ]:
# =============================================================================
# Cell 1 — Setup: path bootstrap + imports
# =============================================================================
import sys, json, time
from pathlib import Path

_ROOT = Path(r"c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat")
sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo

from scripts.paper_2.scenario_runner import (
    load_scenarios_config, _load_yaml, _deep_merge, _dump_yaml_tmp,
    _apply_hp_location, _apply_dsm, _apply_spatial_temperature_offsets,
    _inject_cop_series, _load_outdoor_temps, _load_hp_source_temps,
)
from calion.run.workflow import _build_workflow_inputs
from calion.utils.heizkurve import compute_heizkurve, check_consumer_min_temps
from calion.utils.cop_wrapper import precompute_cop
from calion.models.system_builder import build_model

print("Setup OK. Repo root:", _ROOT)


### Cell 2 — Load scenario definition (MM-S0-HK0)

In [ ]:
# =============================================================================
# Cell 2 — Load scenario definition (MM-S0-HK0)
# =============================================================================
SCEN_ID = "MM-S0-HK0"
scen_cfg = load_scenarios_config()
scen = next(s for s in scen_cfg["scenarios"] if s["id"] == SCEN_ID)
print("Scenario:", json.dumps(scen, indent=2, ensure_ascii=False))


### Cell 3 — Load + merge base config, apply scenario overrides

In [ ]:
# =============================================================================
# Cell 3 — Load + merge base config, apply scenario overrides
# =============================================================================
cfg_path = _ROOT / scen["config"]
cfg = _load_yaml(cfg_path)
if scen.get("overrides"):
    cfg = _deep_merge(cfg, scen["overrides"])
print("tes_main after override:", cfg["assets"]["tes_main"].get("V_min_m3"), cfg["assets"]["tes_main"].get("V_max_m3"))


### Cell 4 — Apply HP/EK location + DSM

In [ ]:
# =============================================================================
# Cell 4 — Apply HP/EK location + DSM
# =============================================================================
# MM-S0-HK0 has tes_node=null -> _apply_tes_location is NOT called (matches
# run_single_scenario's own `if scen.get("tes_node") and not scen.get("baseline")` gate).
if scen.get("hp_node"):
    _apply_hp_location(cfg, scen, scen_cfg)
_apply_dsm(cfg, scen, scen_cfg)  # no-op: dsm_consumers.memmingen == []
print("j_12 assets after HP placement:", cfg["network"]["nodes"]["j_12"].get("assets"))
print("j_1 assets after HP placement:", cfg["network"]["nodes"]["j_1"].get("assets"))


### Cell 5 — Load data table (first pass)

In [ ]:
# =============================================================================
# Cell 5 — Load data table (first pass)
# =============================================================================
tmp1 = _dump_yaml_tmp(cfg)
inputs0 = _build_workflow_inputs([str(tmp1)], overrides=None)
table = inputs0.table
print("Table rows:", len(table), "| dt_h:", inputs0.dt_h)


### Cell 6 — Heating curve T_VL(t) (HK0 stage)

In [ ]:
# =============================================================================
# Cell 6 — Heating curve T_VL(t) (HK0 stage)
# =============================================================================
network = scen["network"]
hk_stage = scen_cfg["heat_curve_stages"][network][scen["heat_curve_stage"]]
print("HK stage:", hk_stage)

T_aus = _load_outdoor_temps(table, cfg)
T_VL_ts = compute_heizkurve(
    k=hk_stage["k"], T_VL_min_c=hk_stage["T_VL_min_c"],
    T_VL_max_c=hk_stage["T_VL_max_c"], T_aus_ts=T_aus,
)

return_temp_c = float(cfg.get("network", {}).get("return_temp_c", 60.0))
min_delta_T = float(cfg.get("network", {}).get("min_supply_delta_T_k", 10.0))
T_VL_min_effective = max(float(hk_stage["T_VL_min_c"]), return_temp_c + min_delta_T)
T_VL_ts = np.maximum(T_VL_ts, T_VL_min_effective)
print(f"T_VL_min_effective={T_VL_min_effective}, T_VL_ts range=[{T_VL_ts.min():.1f}, {T_VL_ts.max():.1f}]")

_consumer_min_c = float(cfg.get("network", {}).get("consumer_min_temp_c", return_temp_c + min_delta_T))
violations = check_consumer_min_temps(T_VL_ts, consumer_min_temps_c={"network": _consumer_min_c})
print("Consumer temp violations:", violations)

cfg.setdefault("network", {}).setdefault("heating_curve", {})
cfg["network"]["heating_curve"]["T_supply_min_c"] = T_VL_min_effective
cfg["network"]["heating_curve"]["T_supply_max_c"] = hk_stage["T_VL_max_c"]
cfg.setdefault("heat_pumps", {}).setdefault("cop", {})
cfg["heat_pumps"]["cop"]["supply_temp_min_c"] = T_VL_min_effective
cfg["heat_pumps"]["cop"]["supply_temp_max_c"] = hk_stage["T_VL_max_c"]

delta_T_scenario_k = round(T_VL_min_effective - return_temp_c, 2)
for _ak, _acfg in cfg.get("assets", {}).items():
    if _acfg.get("type") == "geometric_storage":
        _acfg["delta_T_scenario_k"] = delta_T_scenario_k
print("delta_T_scenario_k:", delta_T_scenario_k)

plt.figure(figsize=(10, 3))
plt.plot(T_VL_ts[:24*14])
plt.title("T_VL(t) -- first 14 days")
plt.ylabel("degC")
plt.tight_layout()


### Cell 7 — Source temp + COP(t) precompute + injection

In [ ]:
# =============================================================================
# Cell 7 — Source temp + COP(t) precompute + injection
# =============================================================================
# Memmingen has no `waste_heat` config section -> falls back to the HP's WRG column.
T_source_ts = _load_hp_source_temps(cfg, table)
cop_ts = precompute_cop(T_VL_ts=T_VL_ts, T_source_ts=T_source_ts, table=table, cfg=cfg, hp_type="standard")
print(f"COP: mean={np.mean(cop_ts):.2f}, min={np.min(cop_ts):.2f}, max={np.max(cop_ts):.2f}")
_inject_cop_series(cfg, cop_ts)

plt.figure(figsize=(10, 3))
plt.plot(cop_ts[:24*14])
plt.title("COP(t) -- first 14 days")
plt.tight_layout()


### Cell 8 — Spatial temperature offsets (real, not skippable)

In [ ]:
# =============================================================================
# Cell 8 — Spatial temperature offsets (real, not skippable)
# =============================================================================
_apply_spatial_temperature_offsets(cfg, T_VL_ts, SCEN_ID)
offsets = {nid: ncfg.get("T_supply_offset_c") for nid, ncfg in cfg["network"]["nodes"].items() if "T_supply_offset_c" in ncfg}
print("Node T_supply_offset_c:", offsets)


### Cell 9 — Final validated config + table load (second, official pass)

In [ ]:
# =============================================================================
# Cell 9 — Final validated config + table load (second, official pass)
# =============================================================================
tmp2 = _dump_yaml_tmp(cfg)
inputs = _build_workflow_inputs([str(tmp2)], overrides=None)
print("Final: table rows =", len(inputs.table), "| dt_h =", inputs.dt_h, "| solver =", inputs.solver_name)
print("hp_main cop_series_override present:",
      "cop_series_override" in inputs.cfg["assets"]["hp_main"],
      "len=", len(inputs.cfg["assets"]["hp_main"].get("cop_series_override", [])))

try:
    tmp1.unlink()
    tmp2.unlink()
except OSError:
    pass

print("\nCELLS 1-9 OK")


### Cell 10 — Build the real Pyomo model (NO SOLVE)

In [ ]:
# =============================================================================
# Cell 10 — Build the real Pyomo model (NO SOLVE)
# =============================================================================
t_build0 = time.perf_counter()
model = build_model(inputs.table, inputs.cfg, dt_h=inputs.dt_h)
t_build = time.perf_counter() - t_build0
print(f"\nModel built in {t_build:.1f} s")

from pyomo.core import Var, Constraint
n_vars = sum(1 for _ in model.component_data_objects(Var, active=True))
n_bin = sum(1 for v in model.component_data_objects(Var, active=True) if v.is_binary())
n_constr = sum(1 for _ in model.component_data_objects(Constraint, active=True))
print(f"n_vars={n_vars:,}  n_bin={n_bin:,}  n_constr={n_constr:,}")

nm = getattr(model, "_network_manager", None)
print("network_manager present:", nm is not None)
if nm is not None:
    print("n_nodes:", len(getattr(nm, "nodes", {})), " n_pipes:", len(getattr(nm, "pipes", {})))

print("_unified_config present:", hasattr(model, "_unified_config"))
print("_system_buses present:", hasattr(model, "_system_buses"))

print("\nCELL 10 OK")


### Cell 11 — OPT-IN SOLVE

In [ ]:
# =============================================================================
# Cell 11 — OPT-IN SOLVE
# The real Paper-2 campaign setting is TimeLimit=86400s (24h), MIPGap=0.005.
# For interactive notebook use, override to something short by default; change
# SOLVE_TIME_LIMIT_S below (and MIPGap) to reproduce the full campaign quality.
# =============================================================================
SOLVE_TIME_LIMIT_S = 240   # short demo value for this test; real campaign = 86400
SOLVE_MIP_GAP = None       # None = use the config's own value (0.005)

solver_options = dict(inputs.cfg.get("run", {}).get("solver_options", {}))
solver_options["TimeLimit"] = SOLVE_TIME_LIMIT_S
if SOLVE_MIP_GAP is not None:
    solver_options["MIPGap"] = SOLVE_MIP_GAP

opt = pyo.SolverFactory(inputs.solver_name)
for key, value in solver_options.items():
    opt.options[key] = value

t_solve0 = time.perf_counter()
solver_result = opt.solve(model, tee=True, warmstart=False, load_solutions=False)
solve_elapsed = time.perf_counter() - t_solve0

solver_meta = {
    "solver_requested": inputs.solver_name,
    "solver_used": inputs.solver_name,
    "status": str(getattr(getattr(solver_result, "solver", None), "status", "unknown")),
    "termination_condition": str(getattr(getattr(solver_result, "solver", None),
                                          "termination_condition", "unknown")),
}
try:
    solver_meta["solution_count"] = len(solver_result.solution)
except Exception:
    solver_meta["solution_count"] = 0
solver_meta["num_vars"] = n_vars
solver_meta["num_bin"] = n_bin
solver_meta["num_constr"] = n_constr
try:
    gb = getattr(opt, "_solver_model", None) or getattr(opt, "_solver", None)
    if gb is not None and hasattr(gb, "MIPGap"):
        solver_meta["mip_gap"] = float(gb.MIPGap)
except Exception:
    pass

print(f"Solved in {solve_elapsed:.1f}s | status={solver_meta['status']} "
      f"| termination={solver_meta['termination_condition']} "
      f"| solutions={solver_meta['solution_count']} | mip_gap={solver_meta.get('mip_gap')}")

if solver_meta["solution_count"] > 0:
    model.solutions.load_from(solver_result)
    print("Solution loaded into model.")

    # Real pipeline calls export_all_results() right here (solver.py:781-814),
    # before result collection -- writes the raw thermal-network parquet/CSV
    # (per-node/per-pipe state) independent of extract_all_p2's own summaries.
    from calion.io.thermal_network_exporter import export_all_results
    export_cfg = inputs.cfg.get("output", {})
    _export_dir = export_cfg.get("export_dir", "output/paper2_runs") + "/thermal_network_results"
    try:
        export_result = export_all_results(
            model=model,
            network_manager=getattr(model, "_network_manager", None),
            time_set=model.t,
            output_dir=_export_dir,
            dt_h=inputs.dt_h,
            export_solver_files=export_cfg.get("export_solver_solution", True),
        )
        solver_meta["export_files"] = export_result.get("files", {})
        solver_meta["export_dir"] = _export_dir  # extract_artefacts_p2 reads node/pipe data from here
        print(f"Thermal-network export: {len(solver_meta['export_files'])} files -> {_export_dir}")
    except Exception as e:
        print(f"[WARN] export_all_results failed (non-fatal): {e}")
else:
    print("[WARN] No incumbent found in the given TimeLimit -- try a longer SOLVE_TIME_LIMIT_S.")


### Cell 12 — Collect results the real way

In [ ]:
# =============================================================================
# Cell 12 — Collect results the real way
# =============================================================================
from calion.run.result_collector import _collect_timeseries_and_summary
from calion.models.results import InvestmentDecisions
from calion.run.types import ScenarioResult, WorkflowResult

model_for_collect = model if solver_meta["solution_count"] > 0 else None
series, summary, costs = _collect_timeseries_and_summary(inputs.table, inputs.cfg, inputs.dt_h, model_for_collect)
investments = InvestmentDecisions.from_summary(summary)
pf_result = ScenarioResult(inputs.table, series, summary, costs, solver_meta, investments)

print("Summary keys:", list(summary.keys())[:15], "...")
print("Objective:", costs.get("objective.OBJ_value_EUR") if isinstance(costs, dict) else None)


### Cell 13 — Export artefacts the real way

In [ ]:
# =============================================================================
# Cell 13 — Export artefacts the real way
# =============================================================================
from scripts.paper_2.extract_artefacts_p2 import extract_all_p2

wf = WorkflowResult(config=inputs.cfg, pf_result=pf_result, rh_result=None,
                     mpc_result=None, design=None, plan=inputs.plan, investments=investments)

NB_OUTDIR = _ROOT / "output" / "paper2_runs" / f"{SCEN_ID}_notebook"
NB_OUTDIR.mkdir(parents=True, exist_ok=True)
extract_all_p2(SCEN_ID, inputs.cfg, wf, solve_elapsed, NB_OUTDIR, scen)

print("Exported files:", sorted(p.name for p in NB_OUTDIR.iterdir()))
print("\nCELLS 11-13 OK")


### Cell 14 — Sanity checks: does this run make sense?

In [ ]:
# =============================================================================
# Cell 14 — Sanity checks: does this run make sense?
# The single most important signal here is heat-balance closure: it's a HARD
# constraint in the MILP, so even a rough/low-quality (high-MIP-gap) solve
# must satisfy it almost exactly -- a large gap means something is genuinely
# wrong (unmet demand), not just "not fully converged".
#
# NOTE on a real discrepancy found while building this cell: the OLD reference
# run at output/paper2_runs/MM-S0-HK0/ only delivers ~6,716 MWh of heat
# against a true annual demand of ~9,433 MWh (model.heatd) -- a ~29% gap,
# with no visible unmet-demand cost line in its economics.csv. A fresh solve
# from this notebook's (current) code closes the same balance to ~0.8%. That
# strongly suggests the stored MM-S0-HK0 reference predates a fix already
# present in the current codebase -- worth confirming/re-running before
# trusting it for anything downstream (e.g. the paper's published numbers).
# =============================================================================
checks = []
def _check(name, ok, detail):
    checks.append({"check": name, "status": "PASS" if ok else "FAIL", "detail": detail})

if solver_meta["solution_count"] == 0:
    print("No incumbent solution -- nothing to sanity-check. Increase SOLVE_TIME_LIMIT_S.")
else:
    dispatch_sc = pd.read_csv(NB_OUTDIR / "dispatch_hourly.csv", parse_dates=["timestamp"])

    # 1) Heat balance closure. Ground truth is model.heatd (what the solver was
    # actually constrained against) -- NOT dispatch_hourly.csv's own
    # "Q_demand_total_MW" column, which was observed to be mislabeled/scaled
    # incorrectly in the export (mean ~73 MW / max ~195 MW for a network whose
    # real peak is ~5-11 MW) and should not be trusted as ground truth.
    demand_MWh = sum(pyo.value(model.heatd[t]) for t in model.t) * inputs.dt_h
    gen_cols = ["Q_chp_MW", "Q_gasboiler_MW", "Q_biomass_MW", "Q_hp_total_MW", "Q_ek_MW"]
    gen_MWh = dispatch_sc[gen_cols].sum().sum() * inputs.dt_h
    gap_pct = 100.0 * (demand_MWh - gen_MWh) / demand_MWh
    _check(
        "Heat balance closes (demand vs dispatched generation)",
        abs(gap_pct) < 3.0,
        f"demand={demand_MWh:.0f} MWh, generated={gen_MWh:.0f} MWh, gap={gap_pct:.1f}% "
        "(should be ~0%; a large gap means unmet demand -- cross-check against "
        "node_heat_audit.json's total_ht_out_MWh)",
    )

    # 2) No significant negative dispatch
    mw_cols = [c for c in dispatch_sc.columns if c.endswith("_MW")]
    neg = {c: int((dispatch_sc[c] < -1e-6).sum()) for c in mw_cols if (dispatch_sc[c] < -1e-6).any()}
    _check("No negative dispatch values", len(neg) == 0, f"columns with negatives: {neg}" if neg else "clean")

    # 3) Temperature ordering: far end can't be hotter than the plant
    if {"T_supply_C", "T_supply_farend_C"}.issubset(dispatch_sc.columns):
        ok = (dispatch_sc["T_supply_farend_C"] <= dispatch_sc["T_supply_C"] + 1e-6).all()
        _check(
            "Far-end supply temp <= plant supply temp", ok,
            f"max violation: {(dispatch_sc['T_supply_farend_C'] - dispatch_sc['T_supply_C']).max():.3f} K",
        )

    # 4) T_supply within the configured heating-curve envelope
    if "T_supply_C" in dispatch_sc.columns:
        lo, hi = T_VL_min_effective - 0.5, hk_stage["T_VL_max_c"] + 0.5
        ok = dispatch_sc["T_supply_C"].between(lo, hi).all()
        _check(
            "T_supply within heating-curve bounds", ok,
            f"range=[{dispatch_sc['T_supply_C'].min():.1f}, {dispatch_sc['T_supply_C'].max():.1f}] "
            f"vs configured [{T_VL_min_effective:.1f}, {hk_stage['T_VL_max_c']:.1f}]",
        )

    # 5) Investable capacities within their configured investment bounds.
    # Real Pyomo Var names -- component naming is NOT consistent across block
    # types (confirmed the hard way: this check originally guessed
    # "eboiler_main_cap_mw" and failed): HeatPumpBlock uses the asset id
    # as-is (component_assembler.py:562-563, "hp_main"), P2HBlock uppercases
    # it (component_assembler.py:766, asset.id.upper() -> "EBOILER_MAIN").
    def _inv_check(label, var_name, asset_key):
        var = getattr(model, var_name, None)
        if var is None:
            _check(f"{label} capacity variable exists", False, f"model.{var_name} not found")
            return
        cap = pyo.value(var)
        inv = inputs.cfg["assets"][asset_key].get("investment", {})
        lo = float(inv.get("capacity_min_mw", 0.0))
        hi = float(inv.get("capacity_max_mw", float("inf")))
        ok = lo - 1e-6 <= cap <= hi + 1e-6
        _check(f"{label} capacity within bounds", ok, f"{cap:.2f} MW, configured [{lo}, {hi}] MW")

    _inv_check("HP (hp_main)", "hp_main_cap_mw", "hp_main")
    _inv_check("EK (eboiler_main)", "EBOILER_MAIN_cap_mw", "eboiler_main")

    tes_var = getattr(model, "tes_main_E_max_expr", None)
    if tes_var is not None:
        tes_e = pyo.value(tes_var)
        _check("TES energy capacity is finite/non-negative", tes_e >= -1e-6, f"{tes_e:.2f} MWh")

    # 6) Objective and solver sanity
    obj_val = pyo.value(model.obj) if hasattr(model, "obj") else None
    _check("Objective is finite and positive", obj_val is not None and obj_val > 0, f"{obj_val}")
    _check(
        "Solver reached a usable incumbent", solver_meta["solution_count"] > 0,
        f"termination={solver_meta['termination_condition']}, mip_gap={solver_meta.get('mip_gap')}",
    )

    checks_df = pd.DataFrame(checks)
    pd.set_option("display.max_colwidth", 100)
    print(checks_df.to_string(index=False))
    n_fail = int((checks_df["status"] == "FAIL").sum())
    print(
        f"\n{len(checks_df) - n_fail}/{len(checks_df)} checks passed."
        + (f"  ({n_fail} FAILED -- see 'detail' column above)" if n_fail else "")
    )

print("\nCELL 14 (sanity checks) OK")


### Cell 15 — Plots (from the real exported dispatch_hourly.csv / economics.csv,

In [ ]:
# =============================================================================
# Cell 15 — Plots (from the real exported dispatch_hourly.csv / economics.csv,
# not hand-extracted from the Pyomo model -- avoids a second transcription risk)
# =============================================================================
dispatch = pd.read_csv(NB_OUTDIR / "dispatch_hourly.csv", parse_dates=["timestamp"], index_col="timestamp")
econ = pd.read_csv(NB_OUTDIR / "economics.csv").iloc[0]

week = dispatch.loc["2025-01-15":"2025-01-22"]
fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(
    week.index,
    week["Q_chp_MW"], week["Q_gasboiler_MW"], week["Q_biomass_MW"],
    week["Q_hp_total_MW"], week["Q_ek_MW"],
    labels=["CHP", "Gasboiler", "Biomass", "Heat pump", "EK"],
)
ax.plot(week.index, week["Q_storage_discharge_MW"], "k--", label="TES discharge", alpha=0.6)
ax.set_ylabel("MW_th")
ax.set_title(f"Memmingen full-network dispatch (real framework) — representative week — {SCEN_ID}")
ax.legend(loc="upper left", ncol=3, fontsize=8)
fig.tight_layout()
fig.savefig(NB_OUTDIR / "dispatch_week.png", dpi=150)
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(dispatch.index, dispatch["T_supply_C"], label="T_supply (plant, j_1)")
ax2.plot(dispatch.index, dispatch["T_supply_farend_C"], label="T_supply (far end)")
ax2.plot(dispatch.index, dispatch["T_return_C"], label="T_return")
ax2.set_ylabel("degC")
ax2.set_title("Network temperatures over the year (real spatial offsets)")
ax2.legend()
fig2.tight_layout()
fig2.savefig(NB_OUTDIR / "temperatures_year.png", dpi=150)
plt.show()

cost_breakdown = pd.Series({
    "Grid buy": econ["cost_energy_buy_eur"],
    "Grid sell": -econ["revenue_sell_eur"],
    "Fuel": econ["cost_fuel_eur"],
    "CO2": econ["cost_co2_eur"],
    "Dump": econ["cost_dump_eur"],
    "Demand charge": econ["cost_demand_charge_eur"],
    "Pump": econ["cost_pump_eur"],
})
fig3, ax3 = plt.subplots(figsize=(7, 4))
cost_breakdown.plot.bar(ax=ax3, color=["#4C72B0" if v >= 0 else "#55A868" for v in cost_breakdown])
ax3.set_ylabel("EUR / a")
ax3.set_title(f"{SCEN_ID} annual cost breakdown (notebook solve)")
fig3.tight_layout()
fig3.savefig(NB_OUTDIR / "cost_breakdown.png", dpi=150)
plt.show()

print("CELL 15 OK")


### Cell 16 — Validation vs. the real reference campaign run

In [ ]:
# =============================================================================
# Cell 16 — Validation vs. the real reference campaign run
# =============================================================================
REF_DIR = _ROOT / "output" / "paper2_runs" / SCEN_ID

ref_meta = json.loads((REF_DIR / "meta.json").read_text(encoding="utf-8"))
nb_meta = json.loads((NB_OUTDIR / "meta.json").read_text(encoding="utf-8"))
ref_econ = pd.read_csv(REF_DIR / "economics.csv").iloc[0]
nb_econ = pd.read_csv(NB_OUTDIR / "economics.csv").iloc[0]

rows = []
def _row(label, ref_v, nb_v, kind="soft"):
    try:
        diff = nb_v - ref_v
        pct = 100.0 * diff / ref_v if ref_v not in (0, None) else float("nan")
    except TypeError:
        diff, pct = None, None
    rows.append({"metric": label, "reference": ref_v, "notebook": nb_v,
                 "abs_diff": diff, "pct_diff": pct, "check": kind})

_row("n_vars (model structure)", ref_meta["n_vars"], nb_meta["n_vars"], "HARD")
_row("n_constraints (model structure)", ref_meta["n_constraints"], nb_meta["n_constraints"], "HARD")
_row("mip_gap", ref_meta.get("mip_gap"), nb_meta.get("mip_gap"), "info")
_row("solve_s", ref_meta.get("solve_s"), nb_meta.get("solve_s"), "info")
_row("obj_eur", ref_meta.get("obj_eur"), nb_meta.get("obj_eur"), "soft (both are time-limited incumbents; "
     "reference is ALSO suspected stale -- see Cell 14 heat-balance note)")
_row("lcoh_eur_per_MWh_th", ref_econ["lcoh_eur_per_MWh_th"], nb_econ["lcoh_eur_per_MWh_th"], "soft")
_row("share_HP_pct", ref_econ["share_HP_pct"], nb_econ["share_HP_pct"], "soft")
_row("share_CHP_pct", ref_econ["share_CHP_pct"], nb_econ["share_CHP_pct"], "soft")
_row("share_EK_pct", ref_econ["share_EK_pct"], nb_econ["share_EK_pct"], "soft")

val_df = pd.DataFrame(rows)
pd.set_option("display.width", 120)
print(val_df.to_string(index=False))

hard_rows = val_df[val_df["check"] == "HARD"]
hard_ok = (hard_rows["abs_diff"].abs() <= 2).all()  # allow tiny incidental diff (n_vars off-by-1 observed)
print(f"\nHard structural fidelity check (n_vars/n_constraints within +/-2 of reference): {'PASS' if hard_ok else 'FAIL'}")
print("Soft cost/dispatch metrics are expected to differ under a short demo TimeLimit AND because the "
      "reference run itself looks stale (see Cell 14) -- treat this table as informational, not a hard gate.")

print("\nCELL 16 OK")
